# TFP-BD 80/20 training: Faster R-CNN (2 GPU)

Amar nijer **80/20 split** (80% train 18,942 / 10% val / 10% test = team er same 2,368 test chobi), nijer data preparation (original XML -> label).

70/20/10 er HUBOHU same recipe: fasterrcnn_resnet50_fpn_v2 (COCO weight), 10 class (9 + background), SGD lr 0.005, momentum 0.9, weight decay 0.0005, StepLR(3, 0.1), **10 epoch, total batch 8**. Shudhu druto korar jonno: 2 GPU (protita GPU te 4 = mot 8), mixed precision.

**Input:** Bangladeshi Traffic Flow Dataset (nijer) + Data Preparation (Sharmin, shudhu common test list). **GPU T4 x2, Internet ON.**

**Cell 3 = SMOKE TEST:** choto 48 ta chobi diye 2 step training + checkpoint + resume check kore. CPU teo chole (GPU quota lage na),
tai GPU dewar AGE Accelerator **None** diye Cell 1-3 chalie "SMOKE OK" dekhe nite paro.

Asol run: GPU T4 x2 -> **Save Version -> Save & Run All**. Somoy: ~6-8 ghonta (2 x T4). Laptop off thakleo cholbe.
Shesh e output e `faster_rcnn_8020_FINAL.pth` thakbe -> Evaluation notebook e ei notebook er output Input hisebe dibe.

In [ ]:
# CELL 1: setup + input khuje ber kora
import os, time, json, glob, shutil, random, subprocess, sys
T0 = time.time()
import torch
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPU:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)] or "(nai - CPU)")
WK = "/kaggle/working"

SH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            SH = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "Bangladeshi Traffic Flow Dataset")]
assert SH, "tfp_dataset pawa jayni - Add Input e Sharmin er 'Data Preparation' notebook jog koro (shudhu test-er file naam er jonno)"
ORIG = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            ORIG = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset")]
assert ORIG, "Original 'Bangladeshi Traffic Flow Dataset' pawa jayni - Add Input e jog koro"
print("Original dataset (nijer):", ORIG)
print("Common test list:", SH + "/test")

# resume: shudhu jodi ager run er output (je khane frcnn_8020_ckpt.pth ache) Input e jog kora thake
RESUME = ""
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset", "Bangladeshi Traffic Flow Dataset", "tfp_8010")]
    if "frcnn_8020_ckpt.pth" in files:
        RESUME = os.path.join(root, "frcnn_8020_ckpt.pth")
print("Resume checkpoint:", RESUME or "nai (notun training shuru hobe)")

In [ ]:
%%writefile /kaggle/working/train_frcnn_8020.py
# Faster R-CNN (ResNet50-FPN v2) on my 80/20 split.
# Same recipe as the team's 70/20/10 run: COCO weights, 10 classes (9 + background), SGD lr 0.005,
# momentum 0.9, weight decay 0.0005, StepLR(step 3, gamma 0.1), 10 epochs, total batch 8, ToTensor only.
# Additions only for speed/safety: 2-GPU DDP (4 images per GPU = same total batch 8, SyncBatchNorm),
# mixed precision, a checkpoint every epoch, and a time limit with resume.
import os, time
import torch
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DATA = os.environ["DATA_ROOT"]
OUT = os.environ["OUT_DIR"]
EPOCHS = int(os.environ.get("EPOCHS", "10"))
BATCH = int(os.environ.get("BATCH", "8"))
DEADLINE = float(os.environ.get("DEADLINE", "1e18"))
RESUME = os.environ.get("RESUME", "")
SMOKE = int(os.environ.get("SMOKE", "0"))
MAX_IT = int(os.environ.get("MAX_IT", "0"))
WORKERS = int(os.environ.get("WORKERS", "2"))
CKPT = os.path.join(OUT, os.environ.get("CKPT_NAME", "frcnn_8020_ckpt.pth"))
FINAL = os.path.join(OUT, os.environ.get("FINAL_NAME", "faster_rcnn_8020_FINAL.pth"))

DDP = int(os.environ.get("WORLD_SIZE", "1")) > 1
CUDA = torch.cuda.is_available()
if DDP:
    dist.init_process_group("nccl" if CUDA else "gloo")
    RANK, WORLD, LOCAL = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", "0"))
else:
    RANK, WORLD, LOCAL = 0, 1, 0
DEV = torch.device(f"cuda:{LOCAL}" if CUDA else "cpu")
if CUDA:
    torch.cuda.set_device(DEV)
MAIN = RANK == 0


def log(*a):
    if MAIN:
        print(*a, flush=True)


class TFBFasterRCNNDataset(Dataset):
    # identical to the 70/20/10 Faster R-CNN notebook
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.imgs = sorted([f for f in os.listdir(img_dir) if f.endswith((".jpg", ".png"))])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        w, h = image.size
        label_path = os.path.join(self.label_dir, img_name.rsplit(".", 1)[0] + ".txt")
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    class_id, cx, cy, bw, bh = map(float, parts)
                    xmin, xmax = max(0, (cx - bw / 2) * w), min(w, (cx + bw / 2) * w)
                    ymin, ymax = max(0, (cy - bh / 2) * h), min(h, (cy + bh / 2) * h)
                    if xmax <= xmin or ymax <= ymin:
                        continue
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(class_id) + 1)
        boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        target = {"boxes": boxes, "labels": torch.as_tensor(labels, dtype=torch.int64), "image_id": torch.tensor([idx])}
        if self.transforms:
            image = self.transforms(image)
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


ds = TFBFasterRCNNDataset(f"{DATA}/train/images", f"{DATA}/train/labels", transforms=T.Compose([T.ToTensor()]))
sampler = DistributedSampler(ds, num_replicas=WORLD, rank=RANK, shuffle=True, seed=0) if DDP else None
PER_GPU = max(1, BATCH // WORLD)
dl = DataLoader(ds, batch_size=PER_GPU, shuffle=sampler is None, sampler=sampler, num_workers=WORKERS,
                pin_memory=CUDA, collate_fn=collate_fn)

model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 10)
start_epoch, ck = 0, None
if RESUME and os.path.exists(RESUME):
    ck = torch.load(RESUME, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    start_epoch = int(ck["epoch"])
    log(f"RESUME: {RESUME} -> {start_epoch} epoch already done")
if DDP and CUDA:
    model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)
model.to(DEV)
net = torch.nn.parallel.DistributedDataParallel(model, device_ids=[LOCAL] if CUDA else None) if DDP else model

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
if ck is not None:
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    lr_scheduler.load_state_dict(ck["lr_scheduler_state_dict"])
    if CUDA and "scaler_state_dict" in ck:
        scaler.load_state_dict(ck["scaler_state_dict"])
    del ck

NB = len(dl)
log(f"train images {len(ds)} | GPUs {WORLD if CUDA else 0} | batch per GPU {PER_GPU} (total {PER_GPU * WORLD}) | "
    f"iterations per epoch {NB} | mixed precision {CUDA} | epochs {start_epoch + 1}..{EPOCHS}")


def state(epoch):
    return {"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
            "lr_scheduler_state_dict": lr_scheduler.state_dict(), "scaler_state_dict": scaler.state_dict()}


stopped, last = False, start_epoch
for epoch in range(start_epoch, EPOCHS):
    if sampler is not None:
        sampler.set_epoch(epoch)
    net.train()
    t_ep, run_loss, n_it = time.time(), 0.0, 0
    for it, (images, targets) in enumerate(dl):
        images = [im.to(DEV, non_blocking=True) for im in images]
        targets = [{k: v.to(DEV) for k, v in t.items()} for t in targets]
        with torch.autocast(device_type="cuda" if CUDA else "cpu", enabled=CUDA):
            loss_dict = net(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += losses.item()
        n_it += 1
        if it == 0 or (it + 1) % 200 == 0:
            el = time.time() - t_ep
            log(f"epoch {epoch + 1}/{EPOCHS}  it {it + 1}/{NB}  loss {losses.item():.4f}  |  {el / 60:.1f} min, "
                f"this epoch ~{el / (it + 1) * NB / 60:.0f} min")
        if MAX_IT and it + 1 >= MAX_IT:
            break
    lr_scheduler.step()
    ep_time = time.time() - t_ep
    last = epoch + 1
    log(f"Epoch {epoch + 1} Completed | Average Loss: {run_loss / max(n_it, 1):.4f} | {ep_time / 60:.1f} min")
    if MAIN:
        torch.save(state(epoch + 1), CKPT)
    stop = torch.tensor([1 if (epoch + 1 < EPOCHS and time.time() + ep_time * 1.1 > DEADLINE) else 0], device=DEV)
    if DDP:
        dist.broadcast(stop, 0)
    if int(stop.item()):
        log(f"TIME LIMIT: {epoch + 1}/{EPOCHS} epoch done, checkpoint saved -> resume needed")
        stopped = True
        break
    if SMOKE and epoch + 1 - start_epoch >= 1:
        break

if MAIN and not stopped and last == EPOCHS:
    torch.save(state(EPOCHS), FINAL)
    log(f"FINAL saved: {FINAL}")
if DDP:
    dist.barrier()
    dist.destroy_process_group()

In [ ]:
# CELL 3: data preparation helper + SMOKE TEST (choto data diye pura pipeline check)

# ---- NIJER data preparation (original XML theke YOLO label) + 80/10/10 split
# test = team er common test set (70/20/10 er same 2,368 chobi, shudhu file-er NAAM Sharmin er folder theke pora hoy)
# train/val = baki 21,310 chobi theke NIJER random split (seed 42): val 2,368 (10%), train 18,942 (80%)
import random, glob
import xml.etree.ElementTree as ET
EXTS = (".jpg", ".jpeg", ".png", ".bmp")
SPLIT_ROOT = "/kaggle/working/tfp_8010"
CLS9 = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
NAME_MAP = {"car": "Car", "bus": "Bus", "truck": "Truck", "bike": "Motorcycle", "cycle": "Bicycle",
            "rickshaw": "Rickshaw", "mini-truck": "Mini-Truck", "cng": "CNG", "people": "Pedestrian"}

def voc_to_yolo_lines(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W, H = float(size.find("width").text), float(size.find("height").text)
    lines = []
    for obj in root.findall("object"):
        key = str(obj.find("name").text).strip().lower()
        if key not in NAME_MAP:
            continue
        b = obj.find("bndbox")
        x0, y0, x1, y1 = [float(b.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")]
        lines.append(f"{CLS9.index(NAME_MAP[key])} {((x0 + x1) / 2) / W:.6f} {((y0 + y1) / 2) / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    return lines

def build_split_8010(SH, out_root=SPLIT_ROOT, make_links=True):
    # 1) original dataset theke image + XML jora
    pairs = {}
    for xf in glob.glob(os.path.join(ORIG, "**", "*.xml"), recursive=True):
        st = os.path.splitext(os.path.basename(xf))[0]
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
            ip = os.path.join(os.path.dirname(xf), st + ext)
            if os.path.exists(ip):
                pairs[st] = (ip, xf)
                break
    # 2) common test set (shudhu naam)
    test = sorted(os.path.splitext(f)[0] for f in os.listdir(f"{SH}/test/images") if f.lower().endswith(EXTS))
    assert all(t in pairs for t in test), "test chobi original dataset e pawa jayni"
    pool = sorted(set(pairs) - set(test))
    order = pool[:]
    random.Random(42).shuffle(order)
    n_val = len(test)
    S = {"train": sorted(order[n_val:]), "val": sorted(order[:n_val]), "test": test}
    if make_links:
        for s, stems in S.items():
            os.makedirs(f"{out_root}/{s}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{s}/labels", exist_ok=True)
            for st in stems:
                ip, xf = pairs[st]
                dst = f"{out_root}/{s}/images/{os.path.basename(ip)}"
                if not os.path.lexists(dst):
                    os.symlink(ip, dst)
                with open(f"{out_root}/{s}/labels/{st}.txt", "w") as fh:
                    fh.write("\n".join(voc_to_yolo_lines(xf)) + "\n")
    tot = sum(len(v) for v in S.values())
    print(f"Original dataset: {len(pairs)} image+XML jora")
    print("Nijer 80/10/10 split:", {k: f"{len(v)} ({len(v) / tot * 100:.1f}%)" for k, v in S.items()})
    # 3) sanity: amar conversion ar Sharmin er label ki eki? (200 ta test file)
    if make_links:
        same, chk = 0, test[:200]
        for st in chk:
            mine = open(f"{out_root}/test/labels/{st}.txt").read().split()
            hers = open(f"{SH}/test/labels/{st}.txt").read().split()
            same += mine == hers
        print(f"  Label conversion check: {same}/{len(chk)} file Sharmin er conversion er sathe hubohu mile")
    return S, pairs

SM = f"{WK}/smoke"
shutil.rmtree(SM, ignore_errors=True)
for d in ("data/train/images", "data/train/labels", "out"):
    os.makedirs(f"{SM}/{d}")
n = 0
for root, dirs, files in os.walk(ORIG):
    for f in sorted(files):
        if not f.endswith(".xml") or n >= 48:
            continue
        st = f[:-4]
        img = next((os.path.join(root, st + e) for e in (".jpg", ".jpeg", ".png") if os.path.exists(os.path.join(root, st + e))), None)
        lines = voc_to_yolo_lines(os.path.join(root, f))
        if img and lines:
            os.symlink(img, f"{SM}/data/train/images/{os.path.basename(img)}")
            open(f"{SM}/data/train/labels/{st}.txt", "w").write("\n".join(lines) + "\n")
            n += 1
    if n >= 48:
        break
print("smoke chobi:", n)
NP_SMOKE = 1 if NGPU == 1 else 2   # CPU te-o 2 process (gloo) diye 2-GPU code path test hoy

def torchrun(env_extra, nproc):
    env = dict(os.environ, **env_extra)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", f"{WK}/train_frcnn_8020.py"]
    return subprocess.run(cmd, env=env).returncode

base = {"DATA_ROOT": f"{SM}/data", "OUT_DIR": f"{SM}/out", "SMOKE": "1", "EPOCHS": "2", "MAX_IT": "2",
        "BATCH": "4", "WORKERS": "0"}
t = time.time()
rc1 = torchrun(base, NP_SMOKE)
assert rc1 == 0 and os.path.exists(f"{SM}/out/frcnn_8020_ckpt.pth"), "SMOKE step 1 FAILED - upore error dekho, screenshot pathao"
rc2 = torchrun(dict(base, RESUME=f"{SM}/out/frcnn_8020_ckpt.pth"), NP_SMOKE)
assert rc2 == 0 and os.path.exists(f"{SM}/out/faster_rcnn_8020_FINAL.pth"), "SMOKE step 2 (resume) FAILED - screenshot pathao"
shutil.rmtree(SM, ignore_errors=True)
print(f"SMOKE OK ({time.time() - t:.0f}s): training, checkpoint ar resume sob kaj kore")

In [ ]:
# CELL 4: amar 80/20 split (original XML theke, symlink - disk e copy hoy na) ~15-20 min
S, SRC = build_split_8010(SH)
json.dump(S, open(f"{WK}/split_8010_lists.json", "w"))
print("train", len(S["train"]), "| val", len(S["val"]), "| test", len(S["test"]))

In [ ]:
# CELL 5: asol training (~6-8 ghonta (2 x T4))
assert NGPU >= 1, "GPU nai - Session options e Accelerator: GPU T4 x2 dao. CPU te training hoy na (shudhu smoke test hoy)."
NP = 2 if NGPU >= 2 else 1
DEADLINE = T0 + 11.2 * 3600          # Kaggle 12 ghonta limit er age checkpoint rekhe thambe
env = {"DATA_ROOT": SPLIT_ROOT, "OUT_DIR": WK, "EPOCHS": "10", "BATCH": "8", "WORKERS": "2", "DEADLINE": str(DEADLINE)}
if RESUME:
    shutil.copy(RESUME, f"{WK}/resume_from.pth")
    env["RESUME"] = f"{WK}/resume_from.pth"
print(f"{NP} GPU diye training shuru | somoy-shima: aro {(DEADLINE - time.time()) / 3600:.1f} ghonta")
rc = torchrun(env, NP)
print("return code:", rc)

In [ ]:
# CELL 6: fol
if os.path.exists(f"{WK}/resume_from.pth"):
    os.remove(f"{WK}/resume_from.pth")
if os.path.exists(f"{WK}/faster_rcnn_8020_FINAL.pth"):
    print("TRAINING SHESH: faster_rcnn_8020_FINAL.pth ready.")
    print("Ekhon Evaluation notebook e ei notebook er output Input hisebe jog kore Save & Run All dao.")
else:
    ep = torch.load(f"{WK}/frcnn_8020_ckpt.pth", map_location="cpu", weights_only=False)["epoch"] if os.path.exists(f"{WK}/frcnn_8020_ckpt.pth") else 0
    print(f"Training shesh hoyni: {ep}/10 epoch hoyeche. Screenshot pathao - resume er niyom bole dibo.")
print("Output:", sorted(f for f in os.listdir(WK) if f.endswith((".pth", ".json"))))